In [1]:
import os
import sys
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

if "google.colab" in sys.modules:
    archive_path = Path("/content/snake_dqn.zip")
    repository_path = Path("/content/snake_dqn")
    urlretrieve(
        "https://github.com/Mazako/snake_dqn/archive/refs/heads/master.zip",
        archive_path,
    )
    with ZipFile(archive_path) as archive:
        archive.extractall(repository_path)

    repository_root = next(path for path in repository_path.iterdir() if path.is_dir())
    os.chdir(repository_root)
    sys.path.insert(0, str(repository_root))

In [2]:
import numpy as np
import torch
from torch import nn


In [3]:
import copy
from random import Random

from snake_dqn.agent import RelativeAction
from snake_dqn.dqn_state import GameState
from snake_dqn.game import Game
from snake_dqn.replay_buffer import ReplayBuffer, Transition


In [4]:
device_type = "cpu"
if torch.cuda.is_available():
    device_type = "cuda"
elif torch.mps.is_available():
    device_type = "mps"

device = torch.device(device_type)
print(f"{device=}")


device=device(type='mps')


In [5]:

seed = 42
rng = Random(seed)
torch.manual_seed(seed)

model = nn.Sequential(
    nn.Linear(12, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 3)
).to(device)
target_model = copy.deepcopy(model).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.SmoothL1Loss()
buffer = ReplayBuffer(10_000, rng)
gamma = 0.99
batch_size = 512
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
learn_every = 16
gradient_steps = 2
target_sync_interval = 250

In [6]:
def select_action(
    model: nn.Module,
    state: GameState,
    epsilon: float,
    rng: Random,
    device: torch.device,
) -> RelativeAction:
    if rng.random() < epsilon:
        return rng.choice(tuple(RelativeAction))

    with torch.no_grad():
        action_index = model(state.features().to(device).unsqueeze(0)).argmax(dim=1).item()

    return RelativeAction(action_index)


def train_step(
    model: nn.Module,
    target_model: nn.Module,
    buffer: ReplayBuffer,
    batch_size: int,
    gamma: float,
    loss_fn: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
) -> float | None:
    if len(buffer) < batch_size:
        return None

    states, actions_batch, rewards, next_states, dones = (
        tensor.to(device) for tensor in buffer.sample(batch_size)
    )

    chosen_q_values = model(states).gather(1, actions_batch.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_q_values = target_model(next_states).max(dim=1).values
        targets = rewards + gamma * (1 - dones) * next_q_values

    loss = loss_fn(chosen_q_values, targets)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    return loss.item()

In [7]:
from collections import deque


def train(
    model: nn.Module,
    target_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    loss_fn: nn.Module,
    buffer: ReplayBuffer,
    rng: Random,
    device: torch.device,
    episodes: int,
    board_size: int,
    max_steps: int,
    gamma: float,
    batch_size: int,
    epsilon: float,
    epsilon_min: float,
    epsilon_decay: float,
    learn_every: int,
    gradient_steps: int,
    target_sync_interval: int,
    log_interval: int,
) -> tuple[list[int], list[float]]:
    global_step = 0
    optimizer_steps = 0
    scores = []
    mean_losses = []
    scores_avgs = deque(maxlen=100)

    for episode in range(episodes):
        game = Game(board_size)
        state = game.game_state()
        episode_losses = []

        for step in range(max_steps):
            action = select_action(model, state, epsilon, rng, device)
            next_state, reward, game_over = game.step(action)
            done = game_over

            buffer.append(Transition(state, action, reward, next_state, done))
            global_step += 1

            if global_step % learn_every == 0:
                for _ in range(gradient_steps):
                    loss = train_step(
                        model,
                        target_model,
                        buffer,
                        batch_size,
                        gamma,
                        loss_fn,
                        optimizer,
                        device,
                    )
                    if loss is not None:
                        episode_losses.append(loss)
                        optimizer_steps += 1

                        if optimizer_steps % target_sync_interval == 0:
                            target_model.load_state_dict(model.state_dict())

            state = next_state

            if done or step == max_steps - 1:
                break

        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        scores.append(game.score)
        scores_avgs.append(game.score)
        mean_loss = (
            np.mean(episode_losses)
            if episode_losses
            else float("nan")
        )
        mean_losses.append(mean_loss)

        if (episode + 1) % log_interval == 0:
            print(
                f"episode={episode + 1} score={game.score} "
                f"epsilon={epsilon:.3f} loss={mean_loss:.4f} "
                f"running avg={np.mean(scores_avgs)}"
            )

    return scores, mean_losses


scores, mean_losses = train(
    model=model,
    target_model=target_model,
    optimizer=optimizer,
    loss_fn=loss_fn,
    buffer=buffer,
    rng=rng,
    device=device,
    episodes=5_000,
    board_size=10,
    max_steps=1_000,
    gamma=gamma,
    batch_size=batch_size,
    epsilon=epsilon,
    epsilon_min=epsilon_min,
    epsilon_decay=epsilon_decay,
    learn_every=learn_every,
    gradient_steps=gradient_steps,
    target_sync_interval=target_sync_interval,
    log_interval=10,
)

episode=10 score=4 epsilon=0.951 loss=0.0284 running avg=4.2
episode=20 score=6 epsilon=0.905 loss=0.0243 running avg=4.35
episode=30 score=4 epsilon=0.860 loss=0.0291 running avg=4.4
episode=40 score=5 epsilon=0.818 loss=0.0378 running avg=4.45
episode=50 score=5 epsilon=0.778 loss=0.0400 running avg=4.46
episode=60 score=5 epsilon=0.740 loss=0.0447 running avg=4.533333333333333
episode=70 score=5 epsilon=0.704 loss=0.0498 running avg=4.628571428571429
episode=80 score=5 epsilon=0.670 loss=0.0475 running avg=4.6875
episode=90 score=4 epsilon=0.637 loss=0.0516 running avg=4.844444444444444
episode=100 score=6 epsilon=0.606 loss=0.0391 running avg=4.91
episode=110 score=7 epsilon=0.576 loss=0.0377 running avg=5.06
episode=120 score=6 epsilon=0.548 loss=0.0400 running avg=5.16
episode=130 score=4 epsilon=0.521 loss=0.0353 running avg=5.26
episode=140 score=8 epsilon=0.496 loss=0.0376 running avg=5.39
episode=150 score=7 epsilon=0.471 loss=0.0390 running avg=5.51
episode=160 score=4 epsil

KeyboardInterrupt: 